In [ ]:
%pip install pandas sqlalchemy pymysql

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import sqlalchemy
import pymysql

print("BingePlay setup ready!")

BingePlay setup ready!


In [ ]:
from sqlalchemy import create_engine, URL
import getpass

password = getpass.getpass("Enter your MySQL password: ")

connection_url = URL.create(
    "mysql+pymysql",
    username="bingeplay_user",
    password=password,
    host="localhost",
    port=3306,
    database="bingeplay"
)

engine = create_engine(connection_url)

with engine.connect() as connection:
    print("Connected to BingePlay successfully!")

Enter your MySQL password:  ········


Connected to BingePlay successfully!


In [ ]:
import subprocess

subprocess.run(["cmd", "/c", "cmdkey", "/list"])

In [ ]:
from sqlalchemy import create_engine, URL

connection_url = URL.create(
    "mysql+pymysql",
    username="bingeplay_user",
    password="BingePlay@2026",
    host="localhost",
    port=3306,
    database="bingeplay"
)

engine = create_engine(connection_url)

with engine.connect() as connection:
    print("Connected to BingePlay successfully!")

In [ ]:
import pandas as pd

query = """
SELECT 'users' AS table_name, COUNT(*) AS total_rows FROM users
UNION ALL
SELECT 'subscriptions', COUNT(*) FROM subscriptions
UNION ALL
SELECT 'shows', COUNT(*) FROM shows
UNION ALL
SELECT 'watch_sessions', COUNT(*) FROM watch_sessions
UNION ALL
SELECT 'ratings', COUNT(*) FROM ratings
"""

pd.read_sql(query, engine)

In [ ]:
users = pd.read_sql("SELECT * FROM users", engine)
subscriptions = pd.read_sql("SELECT * FROM subscriptions", engine)
shows = pd.read_sql("SELECT * FROM shows", engine)
watch_sessions = pd.read_sql("SELECT * FROM watch_sessions", engine)
ratings = pd.read_sql("SELECT * FROM ratings", engine)

print("All BingePlay tables loaded successfully!")
print("Users:", users.shape)
print("Subscriptions:", subscriptions.shape)
print("Shows:", shows.shape)
print("Watch sessions:", watch_sessions.shape)
print("Ratings:", ratings.shape)

In [ ]:
print("Missing values in each table:")

for name, df in {
    "Users": users,
    "Subscriptions": subscriptions,
    "Shows": shows,
    "Watch Sessions": watch_sessions,
    "Ratings": ratings
}.items():
    print(f"\n{name}:")
    print(df.isnull().sum())

In [ ]:
subscriptions["status"].value_counts(dropna=False)

In [ ]:
subscriptions.groupby("status")["end_date"].apply(
    lambda x: x.isnull().sum()
)

In [ ]:
watch_sessions[watch_sessions["user_id"].isnull()]

In [ ]:
watch_sessions.dtypes

In [ ]:
watch_sessions.loc[
    watch_sessions["user_id"].isnull(),
    ["session_id", "user_id", "show_id", "session_date"]
]

In [ ]:
print("Missing show IDs:", watch_sessions["show_id"].isnull().sum())

print("\nAvailable show IDs:")
print(shows[["show_id", "title"]].head(10))

In [ ]:
print(watch_sessions.isnull().sum())

In [ ]:
users["signup_date"] = pd.to_datetime(users["signup_date"])
subscriptions["start_date"] = pd.to_datetime(subscriptions["start_date"])
subscriptions["end_date"] = pd.to_datetime(subscriptions["end_date"])
watch_sessions["session_date"] = pd.to_datetime(watch_sessions["session_date"])
ratings["rated_date"] = pd.to_datetime(ratings["rated_date"])

print("Date columns converted successfully!")

In [ ]:
print("Total watch sessions:", len(watch_sessions))
print("Total watch minutes:", watch_sessions["watch_minutes"].sum())
print("Average watch time (minutes):", round(watch_sessions["watch_minutes"].mean(), 2))
print("Completed sessions:", watch_sessions["completed"].sum())

In [ ]:
completion_rate = (
    watch_sessions["completed"].mean() * 100
)

print("Session completion rate:", round(completion_rate, 2), "%")

In [ ]:
most_watched = (
    watch_sessions
    .groupby("show_id")["session_id"]
    .count()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name="total_sessions")
    .merge(shows[["show_id", "title"]], on="show_id")
)

most_watched[["title", "total_sessions"]]

In [ ]:
category_popularity = (
    watch_sessions
    .merge(shows[["show_id", "category"]], on="show_id")
    .groupby("category")["session_id"]
    .count()
    .sort_values(ascending=False)
    .reset_index(name="total_sessions")
)

category_popularity

In [ ]:
watch_sessions["month"] = (
    watch_sessions["session_date"].dt.to_period("M")
)

monthly_views = (
    watch_sessions.groupby("month")["session_id"]
    .count()
    .reset_index(name="total_sessions")
)

monthly_views

In [ ]:
print("Average rating:", round(ratings["stars"].mean(), 2))
print("\nRating distribution:")
print(ratings["stars"].value_counts().sort_index())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.plot(
    monthly_views["month"].astype(str),
    monthly_views["total_sessions"],
    marker="o"
)

plt.title("BingePlay Monthly Viewing Trend")
plt.xlabel("Month")
plt.ylabel("Total Watch Sessions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    category_popularity["category"],
    category_popularity["total_sessions"]
)

plt.title("BingePlay Viewing by Category")
plt.xlabel("Category")
plt.ylabel("Total Watch Sessions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plan_analysis = subscriptions["plan"].value_counts().reset_index()

plan_analysis.columns = ["plan", "total_subscriptions"]

print(plan_analysis)

In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(
    plan_analysis["plan"],
    plan_analysis["total_subscriptions"]
)

plt.title("BingePlay Subscription Plan Distribution")
plt.xlabel("Subscription Plan")
plt.ylabel("Total Subscriptions")
plt.tight_layout()
plt.show()

In [ ]:
print("BINGEPLAY PROJECT SUMMARY")
print("-------------------------")
print("Total users:", len(users))
print("Total shows:", len(shows))
print("Total watch sessions:", len(watch_sessions))
print("Total subscriptions:", len(subscriptions))
print("Total ratings:", len(ratings))
print("Total watch hours:", round(watch_sessions["watch_minutes"].sum() / 60, 2))
print("Average rating:", round(ratings["stars"].mean(), 2), "/ 5")
print("Session completion rate:", round(watch_sessions["completed"].mean() * 100, 2), "%")

In [ ]:
q1 = """
SELECT
    COUNT(*) AS active_subscriptions,
    SUM(monthly_price_inr) AS total_monthly_revenue
FROM subscriptions
WHERE status = 'active'
  AND (end_date IS NULL OR end_date > '2024-06-30');
"""

q1_result = pd.read_sql(q1, engine)
print(q1_result)

In [ ]:
q2 = """
SELECT
    DATE_FORMAT(signup_date, '%%Y-%%m') AS month,
    COUNT(*) AS signup_count
FROM users
WHERE signup_date >= '2024-01-01'
  AND signup_date < '2024-07-01'
GROUP BY DATE_FORMAT(signup_date, '%%Y-%%m')
ORDER BY month;
"""

q2_result = pd.read_sql(q2, engine)
print(q2_result)

In [ ]:
q3 = """
SELECT
    device_type,
    COUNT(*) AS total_sessions,
    SUM(watch_minutes) AS total_watch_minutes,
    ROUND(AVG(watch_minutes), 2) AS average_watch_minutes,
    ROUND(100.0 * AVG(completed), 2) AS completion_rate
FROM watch_sessions
WHERE user_id IS NOT NULL
GROUP BY device_type
ORDER BY device_type;
"""

q3_result = pd.read_sql(q3, engine)
print(q3_result)

In [ ]:
q4 = """
SELECT
    stars,
    COUNT(*) AS rating_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM ratings), 2)
        AS percentage
FROM ratings
GROUP BY stars
ORDER BY stars;
"""

q4_result = pd.read_sql(q4, engine)
print(q4_result)

q4_high_ratings = pd.read_sql("""
SELECT
    COUNT(*) AS four_or_five_star_count,
    ROUND(
        100.0 * SUM(CASE WHEN stars IN (4, 5) THEN 1 ELSE 0 END)
        / COUNT(*), 2
    ) AS four_or_five_star_percentage
FROM ratings;
""", engine)

print(q4_high_ratings)

In [ ]:
q5 = """
SELECT
    is_original,
    COUNT(*) AS number_of_shows,
    ROUND(AVG(imdb_rating), 2) AS average_imdb_rating,
    ROUND(AVG(release_year), 2) AS average_release_year
FROM shows
GROUP BY is_original
ORDER BY is_original;
"""

q5_result = pd.read_sql(q5, engine)
print(q5_result)

In [ ]:
q6 = """
WITH binge_days AS (
    SELECT
        user_id,
        show_id,
        session_date,
        COUNT(*) AS sessions_in_day
    FROM watch_sessions
    WHERE session_date >= '2024-04-01'
      AND session_date < '2024-07-01'
      AND user_id IS NOT NULL
    GROUP BY user_id, show_id, session_date
    HAVING COUNT(*) >= 5
)
SELECT
    COUNT(*) AS total_binge_days,
    (
        SELECT user_id
        FROM binge_days
        GROUP BY user_id
        ORDER BY COUNT(*) DESC, user_id
        LIMIT 1
    ) AS user_with_most_binge_days,
    (
        SELECT COUNT(*)
        FROM binge_days
        GROUP BY user_id
        ORDER BY COUNT(*) DESC, user_id
        LIMIT 1
    ) AS most_binge_days_by_one_user
FROM binge_days;
"""

q6_result = pd.read_sql(q6, engine)
print(q6_result)

In [ ]:
q7 = """
SELECT
    COUNT(*) AS total_q1_signups,
    SUM(
        CASE
            WHEN NOT EXISTS (
                SELECT 1
                FROM watch_sessions ws
                WHERE ws.user_id = u.user_id
            )
            THEN 1
            ELSE 0
        END
    ) AS users_who_never_watched
FROM users u
WHERE signup_date >= '2024-01-01'
  AND signup_date < '2024-04-01';
"""

q7_result = pd.read_sql(q7, engine)
print(q7_result)

In [ ]:
q8 = """
WITH current_plan AS (
    SELECT user_id, plan
    FROM (
        SELECT
            user_id,
            plan,
            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY start_date DESC, subscription_id DESC
            ) AS rn
        FROM subscriptions
        WHERE status = 'active'
          AND start_date <= '2024-06-30'
          AND (end_date IS NULL OR end_date > '2024-06-30')
    ) ranked
    WHERE rn = 1
)
SELECT COUNT(DISTINCT cp.user_id) AS premium_family_users_only_basic_shows
FROM current_plan cp
WHERE cp.plan IN ('Premium', 'Family')
  AND EXISTS (
      SELECT 1
      FROM watch_sessions ws
      WHERE ws.user_id = cp.user_id
  )
  AND NOT EXISTS (
      SELECT 1
      FROM watch_sessions ws
      JOIN shows s ON s.show_id = ws.show_id
      WHERE ws.user_id = cp.user_id
        AND s.min_plan IN ('Premium', 'Family')
  );
"""

q8_result = pd.read_sql(q8, engine)
print(q8_result)

In [ ]:
q9 = """
WITH first_subscription AS (
    SELECT user_id, plan, start_date,
           ROW_NUMBER() OVER (
               PARTITION BY user_id
               ORDER BY start_date, subscription_id
           ) AS rn
    FROM subscriptions
),
first_upgrade AS (
    SELECT
        fs.user_id,
        u.signup_date,
        MIN(s.start_date) AS upgrade_date
    FROM first_subscription fs
    JOIN subscriptions s ON s.user_id = fs.user_id
    JOIN users u ON u.user_id = fs.user_id
    WHERE fs.rn = 1
      AND fs.plan = 'Basic'
      AND s.plan IN ('Premium', 'Family')
      AND s.start_date > fs.start_date
      AND u.signup_date >= '2024-01-01'
      AND u.signup_date < '2024-02-01'
      AND EXISTS (
          SELECT 1
          FROM subscriptions active_s
          WHERE active_s.user_id = fs.user_id
            AND active_s.status = 'active'
            AND active_s.start_date <= '2024-06-30'
            AND (
                active_s.end_date IS NULL
                OR active_s.end_date > '2024-06-30'
            )
      )
    GROUP BY fs.user_id, u.signup_date
)
SELECT
    COUNT(*) AS upgraded_users,
    ROUND(AVG(DATEDIFF(upgrade_date, signup_date)), 2)
        AS average_days_to_upgrade
FROM first_upgrade;
"""

q9_result = pd.read_sql(q9, engine)
print(q9_result)

In [ ]:
q10 = """
WITH comeback_events AS (
    SELECT DISTINCT
        ws.user_id,
        ws.show_id,
        ws.session_date AS incomplete_date
    FROM watch_sessions ws
    WHERE ws.completed = 0
      AND ws.user_id IS NOT NULL
      AND EXISTS (
          SELECT 1
          FROM watch_sessions later
          WHERE later.user_id = ws.user_id
            AND later.show_id = ws.show_id
            AND later.session_date > ws.session_date
            AND later.session_date <= DATE_ADD(
                ws.session_date, INTERVAL 7 DAY
            )
      )
),
show_counts AS (
    SELECT
        ce.show_id,
        s.title,
        COUNT(*) AS comeback_count
    FROM comeback_events ce
    JOIN shows s ON s.show_id = ce.show_id
    GROUP BY ce.show_id, s.title
)
SELECT
    (SELECT COUNT(*) FROM comeback_events) AS total_comeback_events,
    show_id AS top_show_id,
    title AS top_show_title,
    comeback_count AS top_show_comeback_events
FROM show_counts
ORDER BY comeback_count DESC, show_id
LIMIT 1;
"""

q10_result = pd.read_sql(q10, engine)
print(q10_result)

In [ ]:
q11 = """
WITH weekly_activity AS (
    SELECT DISTINCT
        user_id,
        YEARWEEK(session_date, 3) AS iso_week
    FROM watch_sessions
    WHERE user_id IS NOT NULL
),
numbered_weeks AS (
    SELECT
        user_id,
        iso_week,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY iso_week
        ) AS rn
    FROM weekly_activity
),
week_groups AS (
    SELECT
        user_id,
        iso_week - rn AS streak_group
    FROM numbered_weeks
),
streaks AS (
    SELECT
        user_id,
        streak_group,
        COUNT(*) AS streak_length
    FROM week_groups
    GROUP BY user_id, streak_group
),
qualifying_users AS (
    SELECT user_id, MAX(streak_length) AS longest_streak
    FROM streaks
    GROUP BY user_id
    HAVING MAX(streak_length) >= 4
)
SELECT
    COUNT(*) AS qualifying_users,
    MAX(longest_streak) AS longest_streak_weeks,
    (
        SELECT user_id
        FROM qualifying_users
        WHERE longest_streak = (
            SELECT MAX(longest_streak) FROM qualifying_users
        )
        ORDER BY user_id
        LIMIT 1
    ) AS user_with_longest_streak
FROM qualifying_users;
"""

q11_result = pd.read_sql(q11, engine)
print(q11_result)

In [ ]:
q12 = """
WITH monthly_watch AS (
    SELECT
        user_id,
        SUM(CASE
            WHEN session_date >= '2024-05-01'
             AND session_date < '2024-06-01'
            THEN watch_minutes ELSE 0
        END) AS may_minutes,
        SUM(CASE
            WHEN session_date >= '2024-06-01'
             AND session_date < '2024-07-01'
            THEN watch_minutes ELSE 0
        END) AS june_minutes
    FROM watch_sessions
    WHERE user_id IS NOT NULL
      AND session_date >= '2024-05-01'
      AND session_date < '2024-07-01'
    GROUP BY user_id
)
SELECT
    user_id,
    may_minutes,
    june_minutes,
    ROUND(
        100.0 * (may_minutes - june_minutes) / may_minutes, 2
    ) AS drop_percentage
FROM monthly_watch
WHERE may_minutes > 0
  AND june_minutes <= may_minutes * 0.5
ORDER BY drop_percentage DESC, user_id;
"""

q12_result = pd.read_sql(q12, engine)
print(q12_result)
print("Total users showing churn signal:", len(q12_result))